# Detecting inherent linearity in transformer models

This Notebook serves to experiment with detecting inherent linearity in Llama models. These experiments will be run on Llama-2-7B to test functionality and provide evidence for a feasibility study as part of the graduation preparation phase. Larger experiments for the final paper(s) will be handled using Python files in this repository.

In [1]:
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import login
from datasets import load_dataset
import re
from tqdm import tqdm
import transformers
import os

# Log in to Hugging Face with hf.login file to access the model
login(token=open("../hf.login").read().strip())

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(torch.backends.cuda.matmul.allow_tf32)
print(torch.backends.cudnn.allow_tf32)

CUDA Available: True
GPU: NVIDIA GeForce RTX 4080 SUPER
True
True


In [2]:
# Load the Llama-2-7B model and tokenizer
model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer.pad_token = tokenizer.eos_token # Set pad token to eos token if not already set to prevent errors
print("Model and tokenizer loaded.")

Model and tokenizer loaded.


In [3]:
# Load TinyStories dataset

def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", "", text)  # Remove special characters
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

# Tokenize and clean the dataset
def preprocess(examples):
    examples["text"] = [clean_text(t) for t in examples["text"]]
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)


def load_datasets():
    if not os.path.exists("./data"):
        os.makedirs("./data")

    if os.path.exists("./data/tiny_stories_train") and os.path.exists("./data/tiny_stories_val"):
        train_set = load_dataset("roneneldan/TinyStories", split="train").load_from_disk("./data/tiny_stories_train")
        val_set = load_dataset("roneneldan/TinyStories", split="validation").load_from_disk("./data/tiny_stories_val")
        print("Datasets loaded from disk.")
        return train_set, val_set

    train_set = load_dataset("roneneldan/TinyStories", split="train")
    val_set = load_dataset("roneneldan/TinyStories", split="validation")

    train_set = train_set.map(preprocess, batched=True)
    val_set = val_set.map(preprocess, batched=True)
    print("Datasets loaded and preprocessed.")

    # Save to disk for faster loading later
    train_set.save_to_disk("./data/tiny_stories_train")
    val_set.save_to_disk("./data/tiny_stories_val")
    print("Datasets saved to disk.")

    return train_set, val_set

train_dataset, val_dataset = load_datasets()

Datasets loaded from disk.


In [4]:
# Do a forward pass to ensure everything is working
def forward_pass(model, tokenizer, dataset, device='cuda', debug=False):
    model.to(device)
    model.eval()
    if debug:
        print("Preparing inputs for forward pass...")
    inputs = tokenizer(dataset[0]['text'], return_tensors='pt', truncation=True, padding='max_length', max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if debug:
        print("Inputs prepared. Performing forward pass...")
    with torch.no_grad():
        outputs = model(**inputs)
    if debug:
        print("Forward pass successful.")
    return outputs

# outputs = forward_pass(model, tokenizer, val_dataset, debug=True)
# print("Forward pass output snippet:", outputs.logits[0, :5, :5])

## Visualizing the architecture of Llama-2-7B
To aid with understanding the model architecture, we can visualize the layers and components of Llama-2-7B using a simple diagram.

In [5]:
def visualize_architecture(model):
    for name, module in model.named_modules():
        print(f"{name}: {module.__class__}")

visualize_architecture(model)

: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
model: <class 'transformers.models.llama.modeling_llama.LlamaModel'>
model.embed_tokens: <class 'torch.nn.modules.sparse.Embedding'>
model.layers: <class 'torch.nn.modules.container.ModuleList'>
model.layers.0: <class 'transformers.models.llama.modeling_llama.LlamaDecoderLayer'>
model.layers.0.self_attn: <class 'transformers.models.llama.modeling_llama.LlamaAttention'>
model.layers.0.self_attn.q_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.k_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.v_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.o_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp: <class 'transformers.models.llama.modeling_llama.LlamaMLP'>
model.layers.0.mlp.gate_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.up_proj: <class 'torch.nn.modules.linear.Linear'>
model.layers.0.mlp.down_proj: <class 'torc

## Computing the mean of preactivations per ReLU layer
"**Mean of preactivations $\bar{p}^l$** We denote the distribution of inputs $z$ to a nonlinear activation function $f(z)$ as the preactivations. For each activation function/node, we compute the mean of the preactivations, and then we compute another mean of these values per layer $l$: $\bar{p}^l= \frac{1}{M}\sum_{i=1}^M \left(\frac{1}{N}\sum_{s=1}^N z_{s,i}^l\right)$,
with $M$ the number of nodes in layer $l$ and $N$ the number of samples, and $z_{s,i}^l$ the preactivation value for sample $s$ at node $i$ at layer $l$. We compute this value through randomly selecting 500 samples of the input data instead of the whole dataset, which significantly reduces the computational cost." (Pinson et al., 2024, p. 3).

In case there is BatchNormalization applied between the convolution and the activation function, the mean of the preactivations will be approximately 0, due to the normalization. However, BN has two learned parameters per channel, namely a scaling and a shifting parameter. The shifting parameter can be used to recover the mean of the preactivations before BN. Therefore, in case of BN, we will use the shifting parameter as the mean of the preactivations. (Pinson et al., 2024)

In [6]:
def retrieve_mean_preactivations(model, tokenizer, dataset, device='cuda', save=False):
    """Compute the mean of preactivations for each activation layer in the model. Function does this by computing the mean preactivation values over a set of input data. For Llama with RMS normalization before and after self-attention, we can't retrieve preactivations from normalization parameters. Thus, we must calculate the mean of the input of the normalization before activation named 'model.layers.n.post_attention_layernorm' and 'model.layers.n.mlp.act_fn' where n is the layer number.
    Args:
        model: The neural network model.
        tokenizer: The tokenizer for the model.
        dataset: The dataset to compute preactivations on.
        device: Device to run the computations on.

    Returns:
        dict: A dictionary with layer names as keys and mean preactivation values as values.
    """
    save_path = f"./mean_preactivations_llama2_7b.pt"
    if save and os.path.exists(save_path):
        print("Loading mean preactivations from disk...")
        return torch.load(save_path)

    model.to(device)
    model.eval()
    mean_preactivations = {}
    activation_layers = []
    hooks = []

    print("Identifying activation layers and registering hooks...")
    # Identify activation layers
    for name, module in model.named_modules():
        if re.match(r'model\.layers\.\d+\.mlp\.act_fn', name) or re.match(r'model\.layers\.\d+\.post_attention_layernorm', name):
            activation_layers.append((name, module))
    print("Identified layers, setting hooks...")
    # Define hook to capture preactivations
    def get_preactivation_hook(name):
        def hook(module, input, output):
            if name not in mean_preactivations:
                mean_preactivations[name] = []
            mean_preactivations[name].append(input[0].detach().cpu())
        return hook
    # Register hooks
    for name, module in activation_layers:
        hooks.append(module.register_forward_hook(get_preactivation_hook(name)))
    print("Hooks registered. Performing forward passes...")
    # Forward pass through the data
    with torch.no_grad():
        for i in tqdm(range(2), desc="Processing samples for preactivations", leave=False):
            inputs = tokenizer(dataset[i]['text'], return_tensors='pt', truncation=True, padding='max_length', max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            model(**inputs)
    print("Forward passes complete. Computing mean preactivations...")
    # Compute mean preactivations
    for name in mean_preactivations:
        all_preacts = torch.cat(mean_preactivations[name], dim=0)
        mean_value = all_preacts.mean().item()
        mean_preactivations[name] = mean_value
    print("Mean preactivations computed.")
    # Remove hooks
    for hook in hooks:
        hook.remove()
    print("Hooks removed.")

    if save:
        torch.save(mean_preactivations, save_path)
        print("Mean preactivations saved to disk.")

    return mean_preactivations


mean_preactivations = retrieve_mean_preactivations(model, tokenizer, val_dataset, save=True)
print("Mean preactivations per layer:")
for layer, mean_val in mean_preactivations.items():
    print(f"{layer}: {mean_val}")

Loading mean preactivations from disk...
Mean preactivations per layer:
model.layers.0.post_attention_layernorm: 0.0002632421092130244
model.layers.0.mlp.act_fn: -0.02378932759165764
model.layers.1.post_attention_layernorm: 0.00037288593011908233
model.layers.1.mlp.act_fn: -0.04760121926665306
model.layers.2.post_attention_layernorm: 0.0005880641983821988
model.layers.2.mlp.act_fn: -0.05326581373810768
model.layers.3.post_attention_layernorm: 0.0007293073576875031
model.layers.3.mlp.act_fn: -0.06232980266213417
model.layers.4.post_attention_layernorm: 0.0010481951758265495
model.layers.4.mlp.act_fn: -0.0899086520075798
model.layers.5.post_attention_layernorm: 9.939318988472223e-05
model.layers.5.mlp.act_fn: -0.1174975335597992
model.layers.6.post_attention_layernorm: 0.0009107987862080336
model.layers.6.mlp.act_fn: -0.12449400871992111
model.layers.7.post_attention_layernorm: 0.000526921299751848
model.layers.7.mlp.act_fn: -0.14545613527297974
model.layers.8.post_attention_layernorm: 0

In [7]:
# Map the preactivations of the layernorm to the preceding attention layer for clarity
mapped_mean_preactivations = {}
for layer_name, mean_val in mean_preactivations.items():
    match = re.match(r'model\.layers\.(\d+)\.post_attention_layernorm', layer_name)
    if match:
        layer_num = match.group(1)
        mapped_mean_preactivations[f'model.layers.{layer_num}.self_attn'] = mean_val
        print(f"Mapped model.layers.{layer_num}.self_attn with mean preactivation {mean_val}")

Mapped model.layers.0.self_attn with mean preactivation 0.0002632421092130244
Mapped model.layers.1.self_attn with mean preactivation 0.00037288593011908233
Mapped model.layers.2.self_attn with mean preactivation 0.0005880641983821988
Mapped model.layers.3.self_attn with mean preactivation 0.0007293073576875031
Mapped model.layers.4.self_attn with mean preactivation 0.0010481951758265495
Mapped model.layers.5.self_attn with mean preactivation 9.939318988472223e-05
Mapped model.layers.6.self_attn with mean preactivation 0.0009107987862080336
Mapped model.layers.7.self_attn with mean preactivation 0.000526921299751848
Mapped model.layers.8.self_attn with mean preactivation 0.0004824948264285922
Mapped model.layers.9.self_attn with mean preactivation 0.0007732927333563566
Mapped model.layers.10.self_attn with mean preactivation 0.0011848373105749488
Mapped model.layers.11.self_attn with mean preactivation 0.0013438827591016889
Mapped model.layers.12.self_attn with mean preactivation 0.001

# Utilizing linear approximations based on mean preactivations

To utilize inherent linearity for compression, it will be attempted to approximate the inherently linear sections of a model, based on a thresholded mean preactivation value. This approximation will be a linear layer, that must be trained to mimic the behavior of the a (section of) layer(s) it is approximating. The threshold will be determined based on the distribution of mean preactivation values across layers.

In [8]:
def choose_threshold(mean_preactivations, percentile=25):
    """Choose a threshold based on the given percentile of mean preactivation values."""
    values = list(mean_preactivations.values())
    threshold = np.percentile(values, percentile)
    return threshold

def identify_linear_layers(mean_preactivations, threshold):
    """Identify layers with mean preactivation below the threshold."""
    linear_layers = [layer for layer, mean_val in mean_preactivations.items() if mean_val < threshold]
    return linear_layers

def group_contiguous_layers(linear_layers):
    """
    Groups contiguous model.layers[i].self_attn modules.
    Returns a list of lists of layer indices.
    """
    indices = sorted(
        int(layer.split(".")[2])
        for layer in linear_layers
    )

    groups = []
    current = [indices[0]]

    for prev, curr in zip(indices, indices[1:]):
        if curr == prev + 1:
            current.append(curr)
        else:
            groups.append(current)
            current = [curr]

    groups.append(current)
    return groups

threshold = choose_threshold(mapped_mean_preactivations, percentile=25)
linear_layers = identify_linear_layers(mapped_mean_preactivations, threshold)
groups = group_contiguous_layers(linear_layers)
print(f"Chosen threshold: {threshold}")
print(f"Groups of contiguous linear layers: {groups}")

Chosen threshold: 0.0008764222729951143
Groups of contiguous linear layers: [[0, 1, 2, 3], [5], [7, 8, 9]]


In [9]:
class LinearAttentionBlock(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.linear = torch.nn.Linear(hidden_size, hidden_size)

    def forward(self, hidden_states, **kwargs):
        return self.linear(hidden_states)

class IdentityBlock(torch.nn.Module):
    def forward(self, hidden_states, **kwargs):
        return hidden_states

def replace_attention_block(model, layer_group, linear_block):
    first = layer_group[0]

    # Replace whole forward pass
    model.model.layers[first].forward = linear_block.forward

    # Disable remaining layers
    for layer_id in layer_group[1:]:
        model.model.layers[layer_id].forward = IdentityBlock().forward

def prepare_attention_mask(attention_mask, hidden_states):
    """
    Converts tokenizer attention_mask to a format LLaMA attention accepts.
    """
    # attention_mask: (batch, seq)
    # need shape: (batch, 1, 1, seq) or broadcastable
    return attention_mask[:, None, None, :].to(dtype=torch.bool)


@torch.no_grad()
def get_attention_block_output(model, layer_ids, hidden_states, attention_mask):
    batch_size, seq_len, _ = hidden_states.shape
    device = hidden_states.device

    position_ids = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
    attn_mask = prepare_attention_mask(attention_mask, hidden_states)

    for layer_id in layer_ids:
        layer = model.model.layers[layer_id]

        residual = hidden_states
        normed = layer.input_layernorm(hidden_states)

        position_embeddings = model.model.rotary_emb(normed, position_ids)

        attn_out, _ = layer.self_attn(
            normed,
            attention_mask=attn_mask,
            position_embeddings=position_embeddings,
            past_key_values=None,
            output_attentions=False,
            use_cache=False,
        )

        hidden_states = residual + attn_out

    return hidden_states


def train_block_approximation(
    model,
    tokenizer,
    layer_group,
    train_dataset,
    device,
    epochs=1,
    lr=2e-4,
    max_batches=200
):
    hidden_size = model.config.hidden_size
    approx = LinearAttentionBlock(hidden_size).to(device)

    optimizer = torch.optim.AdamW(approx.parameters(), lr=lr)
    loss_fn = torch.nn.MSELoss()

    model.eval()
    approx.train()

    for epoch in range(epochs):
        for i, batch in tqdm(enumerate(train_dataset), total=min(len(train_dataset), max_batches), desc=f"Training block {layer_group} Epoch {epoch}", leave=False):
            if i >= max_batches:
                break

            inputs = tokenizer(
                batch["text"],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                x = model.model.embed_tokens(inputs.input_ids)
                y_teacher = get_attention_block_output(
                    model,
                    layer_group,
                    x,
                    inputs.attention_mask
                )

            y_student = approx(x)
            loss = loss_fn(y_student, y_teacher)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Block {layer_group} | Epoch {epoch} | Loss {loss.item():.6f}")

    return approx

# Clear memory
try:
    del model
except: # If model is already deleted, we just continue
    pass
torch.cuda.empty_cache()

load_model = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if not load_model:
    compressed_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf").to(device)
    for layer_group in groups:
        print(f"Training approximation for layer group: {layer_group}", flush=True)
        linear_block = train_block_approximation(
            compressed_model,
            tokenizer,
            layer_group,
            train_dataset,
            device,
            epochs=5,
            lr=2e-4,
            max_batches=200,
        )
        replace_attention_block(compressed_model, layer_group, linear_block)

    # Save the compressed model
    compressed_model.save_pretrained("./compressed_llama2_7b")
    print("Compressed model saved to ./compressed_llama2_7b")
else:
    compressed_model = LlamaForCausalLM.from_pretrained("./compressed_llama2_7b").to(device)
    print("Compressed model loaded from ./compressed_llama2_7b")


Compressed model loaded from ./compressed_llama2_7b


### Comparing models
Now that we have the compressed model, let us finetune it for 5 epochs before computing top-5 accuracy loss, size reduction, and inference speedup.

In [ ]:
def finetune_model(model, tokenizer, train_dataset, device, epochs=5, lr=2e-5, max_batches=500):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    model.train()
    for epoch in range(epochs):
        for i, batch in tqdm(enumerate(train_dataset), total=min(len(train_dataset), max_batches), desc=f"Finetuning Epoch {epoch}", leave=False):
            if i >= max_batches:
                break

            inputs = tokenizer(
                batch["text"],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            labels = inputs.input_ids.clone()
            outputs = model(**inputs)
            logits = outputs.logits

            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Finetuning | Epoch {epoch} | Loss {loss.item():.6f}")

    return model

torch.cuda.empty_cache()

# Load the compressed model
compressed_model = LlamaForCausalLM.from_pretrained("./compressed_llama2_7b").to(device)

# Finetune the compressed model
compressed_model = finetune_model(
    compressed_model,
    tokenizer,
    train_dataset,
    device,
    epochs=5,
    lr=2e-5,
    max_batches=50
)

In [10]:
import time
def evaluate_model(model, tokenizer, val_dataset, device, max_batches=200):
    """Evaluate the model on the validation dataset and return average loss and perplexity."""
    model.eval()
    loss_fn = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    total_loss = 0.0
    total_time = 0.0
    total_tokens = 0
    top_5_accuracy = 0
    with torch.no_grad():
        for i, batch in tqdm(enumerate(val_dataset), total=min(len(val_dataset), max_batches), desc="Evaluating", leave=False):
            if i >= max_batches:
                break

            inputs = tokenizer(
                batch["text"],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            labels = inputs.input_ids.clone()

            start_time = time.time()
            outputs = model(**inputs)
            end_time = time.time()

            logits = outputs.logits

            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            num_tokens = (labels != tokenizer.pad_token_id).sum().item()

            total_loss += loss.item() * num_tokens
            total_time += end_time - start_time
            total_tokens += num_tokens
            top_5_accuracy += ((logits.topk(5, dim=-1).indices == labels.unsqueeze(-1)).any(dim=-1) & (labels != tokenizer.pad_token_id)).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss)).item()
    parameter_count = sum(p.numel() for p in model.parameters())
    avg_inference_time = total_time / max_batches
    avg_top_5_accuracy = top_5_accuracy / total_tokens

    return avg_loss, perplexity, parameter_count, avg_inference_time, avg_top_5_accuracy

# Evaluate original model
original_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf").to(device)
original_loss, original_ppl, original_params, original_time, original_top_5_accuracy = evaluate_model(
    original_model,
    tokenizer,
    val_dataset,
    device,
    max_batches=20
)

# Evaluate compressed model
compressed_loss, compressed_ppl, compressed_params, compressed_time, compressed_top_5_accuracy = evaluate_model(
    compressed_model,
    tokenizer,
    val_dataset,
    device,
    max_batches=20
)

# Print results
print("\nEvaluation Results:")
print(f"Original Model - Loss: {original_loss:.4f}, Perplexity: {original_ppl:.2f}, Parameters: {original_params}, Avg Inference Time: {original_time:.4f}s")
print(f"Compressed Model - Loss: {compressed_loss:.4f}, Perplexity: {compressed_ppl:.2f}, Parameters: {compressed_params}, Avg Inference Time: {compressed_time:.4f}s")
size_reduction = (1 - compressed_params / original_params) * 100
speedup = original_time / compressed_time
print(f"Size Reduction: {size_reduction:.2f}%, Inference Speedup: {speedup:.2f}x")
accuracy_loss = original_top_5_accuracy - compressed_top_5_accuracy
print(f"Top-5 Accuracy Loss: {accuracy_loss:.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 15.99 GiB of which 0 bytes is free. Of the allocated memory 29.78 GiB is allocated by PyTorch, and 847.50 KiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)